# Features Engineering

In [1]:
import os
import pandas as pd
import numpy as np
import sqlite3
import calendar

Importing the master dataframe

In [2]:
MASTER_DF_FILEPATH = '../data/processed/flat_files/MASTER_DF.csv'
MASTER_DF = pd.read_csv(MASTER_DF_FILEPATH)
# master copy
df = MASTER_DF.copy()

## 1. Temporal & Calendrical Features
- `days_in_month` , `hours_in_month`, `is_monsoon` 
- `month_sin`, `month_cos` 

### i. Creating `days_in_month` , `hours_in_month`, `is_monsoon` features

In [3]:
# Standard days per calendar month
days_map = { 1: 31, 2: 28, 3: 31, 4: 30, 5: 31, 6: 30, 7: 31, 8: 31, 9: 30, 10: 31, 11: 30, 12: 31 } 

df['days_in_month'] = df['month_num'].map(days_map)


# February having 29 days in a leap year
endyear = (df.fyear.str[:2] + df.fyear.str[-2:]).astype(int)
is_leap = endyear.apply(calendar.isleap)
is_feb = df.month_num == 2
df.loc[is_leap & is_feb, 'days_in_month'] = 29


# Hours in a month
df['hours_in_month'] = df.days_in_month * 24

# Monsoon months
monsoon_months = [7, 8, 9]
df['is_monsoon'] = df.month_num.isin(monsoon_months).astype(int)
df

,state,fyear,month_num,month_name,generation_mu,capacity_mw,peak_demand_mw,solar_irradiance_kwh_m2_day,temperature_2m_c,days_in_month,hours_in_month,is_monsoon
0,Andhra Pradesh,2015-16,4,April,15.44,588.84,6794.0,6.4596,31.19,30,720,0
1,Andhra Pradesh,2015-16,8,August,12.15,588.84,6500.0,5.1350,29.22,31,744,1
2,Andhra Pradesh,2015-16,12,December,31.29,588.84,6544.0,4.5396,24.17,31,744,0
3,Andhra Pradesh,2015-16,2,February,40.32,588.84,7123.0,5.6628,27.60,29,696,0
4,Andhra Pradesh,2015-16,1,January,35.30,588.84,6868.0,4.5756,24.90,31,744,0
...,...,...,...,...,...,...,...,...,...,...,...,...
4345,West Bengal,2025-26,6,June,32.16,320.62,13108.0,4.5168,30.07,30,720,0
4346,West Bengal,2025-26,5,May,40.31,320.62,12693.0,5.6410,31.23,31,744,0
4347,West Bengal,2025-26,11,November,35.34,320.62,9070.0,4.1496,20.32,30,720,0
4348,West Bengal,2025-26,10,October,32.25,320.62,10283.0,4.3255,26.40,31,744,0


### ii. Creating `month_sin`, `month_cos` features
Months are cyclical (December wraps back around to January), so raw numbers (1–12) mislead models into thinking month 12 and month 1 are far apart. 

By projecting the month onto a unit circle using sine and cosine with period $P = 12$:
$$\text{month\_sin} = \sin\left(\frac{2\pi \cdot \text{month}}{12}\right), \quad \text{month\_cos} = \cos\left(\frac{2\pi \cdot \text{month}}{12}\right)$$

* **Preserves continuity:** December (12) and January (1) sit right next to each other in feature space.
* **Prevents overlap:** Using both sine and cosine ensures every month has a unique $(x, y)$ coordinate.

In [4]:
df['month_sin'] = np.sin((2*np.pi*df.month_num)/12)
df['month_cos'] = np.cos((2*np.pi*df.month_num)/12)
df

,state,fyear,month_num,month_name,generation_mu,capacity_mw,peak_demand_mw,solar_irradiance_kwh_m2_day,temperature_2m_c,days_in_month,hours_in_month,is_monsoon,month_sin,month_cos
0,Andhra Pradesh,2015-16,4,April,15.44,588.84,6794.0,6.4596,31.19,30,720,0,8.660254e-01,-5.000000e-01
1,Andhra Pradesh,2015-16,8,August,12.15,588.84,6500.0,5.1350,29.22,31,744,1,-8.660254e-01,-5.000000e-01
2,Andhra Pradesh,2015-16,12,December,31.29,588.84,6544.0,4.5396,24.17,31,744,0,-2.449294e-16,1.000000e+00
3,Andhra Pradesh,2015-16,2,February,40.32,588.84,7123.0,5.6628,27.60,29,696,0,8.660254e-01,5.000000e-01
4,Andhra Pradesh,2015-16,1,January,35.30,588.84,6868.0,4.5756,24.90,31,744,0,5.000000e-01,8.660254e-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4345,West Bengal,2025-26,6,June,32.16,320.62,13108.0,4.5168,30.07,30,720,0,1.224647e-16,-1.000000e+00
4346,West Bengal,2025-26,5,May,40.31,320.62,12693.0,5.6410,31.23,31,744,0,5.000000e-01,-8.660254e-01
4347,West Bengal,2025-26,11,November,35.34,320.62,9070.0,4.1496,20.32,30,720,0,-5.000000e-01,8.660254e-01
4348,West Bengal,2025-26,10,October,32.25,320.62,10283.0,4.3255,26.40,31,744,0,-8.660254e-01,5.000000e-01


---

## 2. Physics & Weather Features
- `pr_temp_adjusted`
- `potential_generation_mu`

### i. Creating `pr_temp_adjusted` feature

- `pr_temp_adjusted` represents the temperature-adjusted Performance Ratio (PR) of a silicon photovoltaic (PV) cell.

Silicon photovoltaic cells are rated for optimal performance at a reference temperature of $25^\circ\text{C}$. The standard industry Performance Ratio ($PR$) is assumed to be $0.75$.

For every degree Celsius increase above the reference temperature, the Performance Ratio decreases by $0.4\%$.

Therefore, the temperature-adjusted Performance Ratio is calculated as:

$$
PR_{\text{temp-adjusted}} = 0.75 \times \left[1 - 0.004 \times (T - 25)\right]
$$

where:
- $T$ = Ambient temperature in $^\circ\text{C}$.
- $25$ = Reference temperature in $^\circ\text{C}$.
- $0.004$ = Temperature derating coefficient ($0.4\%$ per $^\circ\text{C}$).
- $0.75$ = Standard industry Performance Ratio.

In [5]:
BASE_PR = 0.75
TEMP_COEFF = 0.004
# Creating a pandas series for temp_excess or T - 25
temp_excess = np.maximum(0, df.temperature_2m_c - 25)

df['pr_temp_adjusted'] = BASE_PR * (1 - TEMP_COEFF * temp_excess)

df

,state,fyear,month_num,month_name,generation_mu,capacity_mw,peak_demand_mw,solar_irradiance_kwh_m2_day,temperature_2m_c,days_in_month,hours_in_month,is_monsoon,month_sin,month_cos,pr_temp_adjusted
0,Andhra Pradesh,2015-16,4,April,15.44,588.84,6794.0,6.4596,31.19,30,720,0,8.660254e-01,-5.000000e-01,0.73143
1,Andhra Pradesh,2015-16,8,August,12.15,588.84,6500.0,5.1350,29.22,31,744,1,-8.660254e-01,-5.000000e-01,0.73734
2,Andhra Pradesh,2015-16,12,December,31.29,588.84,6544.0,4.5396,24.17,31,744,0,-2.449294e-16,1.000000e+00,0.75000
3,Andhra Pradesh,2015-16,2,February,40.32,588.84,7123.0,5.6628,27.60,29,696,0,8.660254e-01,5.000000e-01,0.74220
4,Andhra Pradesh,2015-16,1,January,35.30,588.84,6868.0,4.5756,24.90,31,744,0,5.000000e-01,8.660254e-01,0.75000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4345,West Bengal,2025-26,6,June,32.16,320.62,13108.0,4.5168,30.07,30,720,0,1.224647e-16,-1.000000e+00,0.73479
4346,West Bengal,2025-26,5,May,40.31,320.62,12693.0,5.6410,31.23,31,744,0,5.000000e-01,-8.660254e-01,0.73131
4347,West Bengal,2025-26,11,November,35.34,320.62,9070.0,4.1496,20.32,30,720,0,-5.000000e-01,8.660254e-01,0.75000
4348,West Bengal,2025-26,10,October,32.25,320.62,10283.0,4.3255,26.40,31,744,0,-8.660254e-01,5.000000e-01,0.74580


**Observations** : The mean adjusted performance ratio is approximately 0.7442, close to the baseline value of 0.75. The relatively low standard deviation (0.0082) indicates limited variability across observations. The median and 75th percentile are both 0.75, suggesting that at least 75% of observations retain the baseline performance ratio. The negative skewness (-1.47) indicates a longer tail towards lower values, reflecting temperature-induced performance reductions in a subset of observations. The maximum of 0.75 confirms that temperature adjustment only reduces the baseline performance ratio, never increases it.

### ii. Creating `potential_generation_mu`
- `potential_generation_mu` represents the theoretical maximum solar electricity generation (in Million Units, MU) that the installed photovoltaic (PV) panels could produce under the given solar irradiance and temperature conditions.

It is calculated using the installed solar capacity, daily solar irradiance, number of days in the period, and temperature-adjusted Performance Ratio.

The potential generation is calculated as:

$$
\text{Potential Generation (MU)} =
\frac{
Capacity \times Irradiance \times Days \times PR_{\text{temp-adjusted}}
}{1000}
$$

where:
- $Capacity$ = Installed solar capacity (MW).
- $Irradiance$ = Daily solar irradiance (kWh/m²/day).
- $Days$ = Number of days in the period.
- $Performance Ratio_{\text{temp-adjusted}}$ = Temperature-adjusted Performance Ratio.
- $1000$ = Conversion factor from MWh to MU, since $1\text{ MU} = 1000\text{ MWh}$.

In [6]:
df['potential_generation_mu'] = (df.capacity_mw * df.solar_irradiance_kwh_m2_day * df.days_in_month * df.pr_temp_adjusted)/1000
df

,state,fyear,month_num,month_name,generation_mu,capacity_mw,peak_demand_mw,solar_irradiance_kwh_m2_day,temperature_2m_c,days_in_month,hours_in_month,is_monsoon,month_sin,month_cos,pr_temp_adjusted,potential_generation_mu
0,Andhra Pradesh,2015-16,4,April,15.44,588.84,6794.0,6.4596,31.19,30,720,0,8.660254e-01,-5.000000e-01,0.73143,83.463569
1,Andhra Pradesh,2015-16,8,August,12.15,588.84,6500.0,5.1350,29.22,31,744,1,-8.660254e-01,-5.000000e-01,0.73734,69.114193
2,Andhra Pradesh,2015-16,12,December,31.29,588.84,6544.0,4.5396,24.17,31,744,0,-2.449294e-16,1.000000e+00,0.75000,62.149530
3,Andhra Pradesh,2015-16,2,February,40.32,588.84,7123.0,5.6628,27.60,29,696,0,8.660254e-01,5.000000e-01,0.74220,71.770748
4,Andhra Pradesh,2015-16,1,January,35.30,588.84,6868.0,4.5756,24.90,31,744,0,5.000000e-01,8.660254e-01,0.75000,62.642389
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4345,West Bengal,2025-26,6,June,32.16,320.62,13108.0,4.5168,30.07,30,720,0,1.224647e-16,-1.000000e+00,0.73479,31.923166
4346,West Bengal,2025-26,5,May,40.31,320.62,12693.0,5.6410,31.23,31,744,0,5.000000e-01,-8.660254e-01,0.73131,41.002460
4347,West Bengal,2025-26,11,November,35.34,320.62,9070.0,4.1496,20.32,30,720,0,-5.000000e-01,8.660254e-01,0.75000,29.935007
4348,West Bengal,2025-26,10,October,32.25,320.62,10283.0,4.3255,26.40,31,744,0,-8.660254e-01,5.000000e-01,0.74580,32.063505


---

## 3. Grid Economics & Curtailment Metric Features
- `cuf_actual` 
- `curtailement_proxy_mu` & `curtailment_ratio` 
- `solar-penetration-ratio` & `solar-surplus-risk` 

### i. Creating `cuf_actual` feature
- `cuf_actual` represents the actual Capacity Utilization Factor (CUF) of the installed solar capacity during the concerned month. It indicates how effectively the installed capacity was utilized to generate electricity relative to its maximum possible generation over the same period.

The actual CUF is calculated as:

$$
CUF_{\text{actual}} =
\frac{
\text{Generation (MU)} \times 1000
}{
\text{Capacity (MW)} \times H
}
\times 100
$$

where:
- $\text{Generation (MU)}$ = Actual electricity generated during the month, in Million Units.
- $\text{Capacity (MW)}$ = Installed solar capacity, in megawatts.
- $H$ = Total number of hours in the concerned month.
- $100$ = Conversion factor to express CUF as a percentage.

Since $1\text{ MU} = 1000\text{ MWh}$, the generation is converted from MU to MWh before calculating CUF.

In [7]:
df['cuf_actual'] = ((df.generation_mu * 1000)/(df.capacity_mw * df.hours_in_month)) * 100
df

,state,fyear,month_num,month_name,generation_mu,capacity_mw,peak_demand_mw,solar_irradiance_kwh_m2_day,temperature_2m_c,days_in_month,hours_in_month,is_monsoon,month_sin,month_cos,pr_temp_adjusted,potential_generation_mu,cuf_actual
0,Andhra Pradesh,2015-16,4,April,15.44,588.84,6794.0,6.4596,31.19,30,720,0,8.660254e-01,-5.000000e-01,0.73143,83.463569,3.641812
1,Andhra Pradesh,2015-16,8,August,12.15,588.84,6500.0,5.1350,29.22,31,744,1,-8.660254e-01,-5.000000e-01,0.73734,69.114193,2.773359
2,Andhra Pradesh,2015-16,12,December,31.29,588.84,6544.0,4.5396,24.17,31,744,0,-2.449294e-16,1.000000e+00,0.75000,62.149530,7.142255
3,Andhra Pradesh,2015-16,2,February,40.32,588.84,7123.0,5.6628,27.60,29,696,0,8.660254e-01,5.000000e-01,0.74220,71.770748,9.838162
4,Andhra Pradesh,2015-16,1,January,35.30,588.84,6868.0,4.5756,24.90,31,744,0,5.000000e-01,8.660254e-01,0.75000,62.642389,8.057577
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4345,West Bengal,2025-26,6,June,32.16,320.62,13108.0,4.5168,30.07,30,720,0,1.224647e-16,-1.000000e+00,0.73479,31.923166,13.931341
4346,West Bengal,2025-26,5,May,40.31,320.62,12693.0,5.6410,31.23,31,744,0,5.000000e-01,-8.660254e-01,0.73131,41.002460,16.898543
4347,West Bengal,2025-26,11,November,35.34,320.62,9070.0,4.1496,20.32,30,720,0,-5.000000e-01,8.660254e-01,0.75000,29.935007,15.308881
4348,West Bengal,2025-26,10,October,32.25,320.62,10283.0,4.3255,26.40,31,744,0,-8.660254e-01,5.000000e-01,0.74580,32.063505,13.519673


**Observations** : The mean actual capacity utilisation factor (CUF) is approximately 8.98%, with a median of 7.43%, indicating generally low utilisation across observations. The standard deviation (13.81 percentage points) suggests substantial variability. The 75th percentile is 13.68%, while the maximum reaches 446.90%, indicating extreme outliers or potentially anomalous observations. The very high positive skewness (14.99) and kurtosis (354.69) confirm a heavily right-skewed distribution, with a small number of exceptionally high values. These extreme observations warrant further investigation before drawing conclusions about actual capacity utilisation.

### ii. Creating `curtailement_proxy_mu` & `curtailment_ratio` features

- `curtailment_proxy_mu` represents the estimated difference between the potential solar electricity generation and the actual electricity generated during the concerned month. It provides an estimate of the solar generation that could potentially have been realized under the assumed irradiance and performance conditions but was not reflected in actual generation.

The curtailment proxy is calculated as:

$$
\text{Curtailment Proxy (MU)} =
\text{Potential Generation (MU)}
-
\text{Actual Generation (MU)}
$$

where:
- $\text{Potential Generation (MU)}$ = Estimated electricity generation based on installed capacity, solar irradiance, and temperature-adjusted Performance Ratio.
- $\text{Actual Generation (MU)}$ = Observed electricity generated during the month.

<br>
<br>
<br>

- `curtailment_ratio` represents the proportion of estimated potential solar generation that was not realized as actual electricity generation. It expresses the estimated generation shortfall as a fraction of the total potential generation.

The curtailment ratio is calculated as:

$$
\text{Curtailment Ratio} =
\frac{
\text{Curtailment Proxy (MU)}
}{
\text{Potential Generation (MU)}
}
$$

where:
- $\text{Curtailment Proxy (MU)}$ = Difference between potential and actual generation.
- $\text{Potential Generation (MU)}$ = Estimated maximum generation under the assumed solar irradiance and performance conditions.

A higher curtailment ratio indicates a larger gap between estimated potential and actual generation.

**TL;DR** : `curtailment_ratio` gives an estimate of the unharvested solar generation lost due to grid backing down directives, transmission congestion, plant downtime, etc. 

**Note** : The curtailment ratio represents the non-negative estimated generation shortfall as a proportion of potential generation. Since curtailment_proxy_mu is clipped at zero, observations where actual generation exceeds estimated potential generation are assigned a ratio of zero. This is a generation shortfall proxy, not a direct measurement of grid-induced curtailment.

In [8]:
#Curtailment proxy 
df['curtailment_proxy_mu'] = np.maximum(0.0, df.potential_generation_mu - df.generation_mu)

#Curtailment ratio
df['curtailment_ratio'] = df.curtailment_proxy_mu/df.potential_generation_mu

df

,state,fyear,month_num,month_name,generation_mu,capacity_mw,peak_demand_mw,solar_irradiance_kwh_m2_day,temperature_2m_c,days_in_month,hours_in_month,is_monsoon,month_sin,month_cos,pr_temp_adjusted,potential_generation_mu,cuf_actual,curtailment_proxy_mu,curtailment_ratio
0,Andhra Pradesh,2015-16,4,April,15.44,588.84,6794.0,6.4596,31.19,30,720,0,8.660254e-01,-5.000000e-01,0.73143,83.463569,3.641812,68.023569,0.815009
1,Andhra Pradesh,2015-16,8,August,12.15,588.84,6500.0,5.1350,29.22,31,744,1,-8.660254e-01,-5.000000e-01,0.73734,69.114193,2.773359,56.964193,0.824204
2,Andhra Pradesh,2015-16,12,December,31.29,588.84,6544.0,4.5396,24.17,31,744,0,-2.449294e-16,1.000000e+00,0.75000,62.149530,7.142255,30.859530,0.496537
3,Andhra Pradesh,2015-16,2,February,40.32,588.84,7123.0,5.6628,27.60,29,696,0,8.660254e-01,5.000000e-01,0.74220,71.770748,9.838162,31.450748,0.438211
4,Andhra Pradesh,2015-16,1,January,35.30,588.84,6868.0,4.5756,24.90,31,744,0,5.000000e-01,8.660254e-01,0.75000,62.642389,8.057577,27.342389,0.436484
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4345,West Bengal,2025-26,6,June,32.16,320.62,13108.0,4.5168,30.07,30,720,0,1.224647e-16,-1.000000e+00,0.73479,31.923166,13.931341,0.000000,0.000000
4346,West Bengal,2025-26,5,May,40.31,320.62,12693.0,5.6410,31.23,31,744,0,5.000000e-01,-8.660254e-01,0.73131,41.002460,16.898543,0.692460,0.016888
4347,West Bengal,2025-26,11,November,35.34,320.62,9070.0,4.1496,20.32,30,720,0,-5.000000e-01,8.660254e-01,0.75000,29.935007,15.308881,0.000000,0.000000
4348,West Bengal,2025-26,10,October,32.25,320.62,10283.0,4.3255,26.40,31,744,0,-8.660254e-01,5.000000e-01,0.74580,32.063505,13.519673,0.000000,0.000000


**Observations**: 
- `curtailment_proxy` : The mean estimated generation shortfall (curtailment proxy) is approximately 26.27, considerably higher than the median of 1.18, indicating a strongly right-skewed distribution. The standard deviation (115.30) suggests substantial variability across observations. The 75th percentile is 9.88, while the maximum reaches 2,327.32, indicating the presence of extreme shortfall values. The large difference between the mean and median suggests that a relatively small number of observations account for a substantial proportion of the estimated generation shortfall.
- `curtailment_ratio` : The mean curtailment ratio is approximately 49.2%, with a median of 48.5%, indicating a substantial estimated generation shortfall across observations. The wide standard deviation (38.5 percentage points) suggests considerable variability. The middle 50% of observations range from 6.6% to 88.3%, indicating a highly dispersed distribution. Ratios span the full 0–100% range, suggesting that estimated shortfalls vary significantly across observations.

### iii. Creating `solar-penetration-ratio` & `solar-surplus-risk` features

- `solar-penetration-ratio` (also known as the **grid penetration ratio** or **supply stress ratio**) measures the proportion of a state's solar fleet relative to local peak electricity demand, indicating potential export stress on interstate transmission lines.

It is calculated as:

$$
\text{Solar Penetration Ratio} = \frac{\text{Capacity (MW)}}{\text{Peak Demand (MW)}}
$$

where:
- $\text{Capacity (MW)}$ = Installed solar capacity, in megawatts.
- $\text{Peak Demand (MW)}$ = Local peak electricity demand, in megawatts.

A ratio greater than $1$ indicates that installed solar capacity exceeds local peak demand, creating export pressure on interstate transmission infrastructure.

- `solar-surplus-risk` (also known as the **absorption cushion**) quantifies the absolute capacity margin by which installed solar generation exceeds or falls short of local peak demand.

It is calculated as:

$$
\text{Solar Surplus Risk} = \text{Capacity (MW)} - \text{Peak Demand (MW)}
$$

where:
- $\text{Capacity (MW)}$ = Installed solar capacity, in megawatts.
- $\text{Peak Demand (MW)}$ = Local peak electricity demand, in megawatts.

A positive value reflects excess solar capacity beyond local peak demand (surplus risk), while a negative value indicates remaining local absorption headroom.

In [9]:
# Solar penetration ratio
df['solar_penetration_ratio'] = df.capacity_mw/df.peak_demand_mw

# Solar surplus risk
df['solar-surplus-risk'] = df.capacity_mw - df.peak_demand_mw

df

,state,fyear,month_num,month_name,generation_mu,capacity_mw,peak_demand_mw,solar_irradiance_kwh_m2_day,temperature_2m_c,days_in_month,...,is_monsoon,month_sin,month_cos,pr_temp_adjusted,potential_generation_mu,cuf_actual,curtailment_proxy_mu,curtailment_ratio,solar_penetration_ratio,solar-surplus-risk
0,Andhra Pradesh,2015-16,4,April,15.44,588.84,6794.0,6.4596,31.19,30,...,0,8.660254e-01,-5.000000e-01,0.73143,83.463569,3.641812,68.023569,0.815009,0.086671,-6205.16
1,Andhra Pradesh,2015-16,8,August,12.15,588.84,6500.0,5.1350,29.22,31,...,1,-8.660254e-01,-5.000000e-01,0.73734,69.114193,2.773359,56.964193,0.824204,0.090591,-5911.16
2,Andhra Pradesh,2015-16,12,December,31.29,588.84,6544.0,4.5396,24.17,31,...,0,-2.449294e-16,1.000000e+00,0.75000,62.149530,7.142255,30.859530,0.496537,0.089982,-5955.16
3,Andhra Pradesh,2015-16,2,February,40.32,588.84,7123.0,5.6628,27.60,29,...,0,8.660254e-01,5.000000e-01,0.74220,71.770748,9.838162,31.450748,0.438211,0.082667,-6534.16
4,Andhra Pradesh,2015-16,1,January,35.30,588.84,6868.0,4.5756,24.90,31,...,0,5.000000e-01,8.660254e-01,0.75000,62.642389,8.057577,27.342389,0.436484,0.085737,-6279.16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4345,West Bengal,2025-26,6,June,32.16,320.62,13108.0,4.5168,30.07,30,...,0,1.224647e-16,-1.000000e+00,0.73479,31.923166,13.931341,0.000000,0.000000,0.024460,-12787.38
4346,West Bengal,2025-26,5,May,40.31,320.62,12693.0,5.6410,31.23,31,...,0,5.000000e-01,-8.660254e-01,0.73131,41.002460,16.898543,0.692460,0.016888,0.025260,-12372.38
4347,West Bengal,2025-26,11,November,35.34,320.62,9070.0,4.1496,20.32,30,...,0,-5.000000e-01,8.660254e-01,0.75000,29.935007,15.308881,0.000000,0.000000,0.035350,-8749.38
4348,West Bengal,2025-26,10,October,32.25,320.62,10283.0,4.3255,26.40,31,...,0,-8.660254e-01,5.000000e-01,0.74580,32.063505,13.519673,0.000000,0.000000,0.031180,-9962.38


**Observations**: 
- `solar-penetration-ratio` : The mean solar penetration ratio is approximately 0.160 (16.0%), with a median of 0.0665 (6.65%), indicating that installed solar capacity is generally modest relative to state peak electricity demand. The standard deviation (0.245) suggests considerable variability across observations. The 75th percentile is 0.1914 (19.14%), while the maximum reaches 2.767 (276.7%), indicating that some observations have installed solar capacity substantially exceeding peak demand. The high positive skewness (3.35) and kurtosis (16.57) indicate a heavily right-skewed distribution with extreme high-value observations. These observations may represent states with particularly high solar penetration and potential dependence on interstate transmission infrastructure.
Solar Surplus Risk
- `solar-surplus-risk` : The mean solar surplus risk is approximately −6,241.60 MW, with a median of −3,305.12 MW, indicating that installed solar capacity is generally below state peak electricity demand. The standard deviation (11,580.71 MW) suggests substantial variation across observations. The 75th percentile is −300 MW, meaning that at least 75% of observations have installed solar capacity below peak demand. However, the maximum reaches 25,212.62 MW, indicating that some observations have substantial installed capacity exceeding peak demand. The strong negative skewness (−4.18) and high kurtosis (20.25) indicate a distribution with a pronounced tail towards large negative values and extreme observations. These positive surplus-risk observations warrant further investigation to identify states where installed solar capacity exceeds local peak demand.

---

## 4. Time Series & Rolling Metrics (Per State)
    i. Creating `gen_roll3_mu` & `curtailment_roll3_mu` features

Three-month rolling averages are calculated separately for actual solar generation and estimated generation shortfall, grouped by state.

The rolling averages are calculated using the preceding three observations within each state, with `min_periods=1` allowing calculations to begin even when fewer than three observations are available.

This transformation aims to:
- Smooth short-term fluctuations in monthly generation and estimated shortfall.
- Capture underlying trends in solar generation and potential curtailment.
- Reduce the influence of isolated monthly variations when analysing state-level patterns.

In [10]:
#Sorting each observations within each state chronologically.
month_fy_order = {4:1, 5:2, 6:3, 7:4, 8:5, 9:6, 10:7, 11:8, 12:9, 1:10, 2:11, 3:12} 
df['fy_month_order'] = df['month_num'].map(month_fy_order) 
df = df.sort_values(by=['state', 'fyear', 'fy_month_order']).reset_index(drop=True)

In [11]:
#3 Month Rolling Average of Generation per state
df['gen_roll3_mu'] = df.groupby('state')['generation_mu'].transform(lambda s: s.rolling(3, min_periods=1).mean())

#3 Month Rolling Average of Curtailment per state
df['curtailment_roll3_mu'] = df.groupby('state')['curtailment_proxy_mu'].transform(lambda s: s.rolling(3, min_periods=1).mean())

df

,state,fyear,month_num,month_name,generation_mu,capacity_mw,peak_demand_mw,solar_irradiance_kwh_m2_day,temperature_2m_c,days_in_month,...,pr_temp_adjusted,potential_generation_mu,cuf_actual,curtailment_proxy_mu,curtailment_ratio,solar_penetration_ratio,solar-surplus-risk,fy_month_order,gen_roll3_mu,curtailment_roll3_mu
0,Andhra Pradesh,2015-16,4,April,15.44,588.84,6794.0,6.4596,31.19,30,...,0.73143,83.463569,3.641812,68.023569,0.815009,0.086671,-6205.16,1,15.440000,68.023569
1,Andhra Pradesh,2015-16,5,May,13.35,588.84,6732.0,6.4200,34.16,31,...,0.72252,84.672796,3.047271,71.322796,0.842334,0.087469,-6143.16,2,14.395000,69.673183
2,Andhra Pradesh,2015-16,6,June,10.76,588.84,6628.0,4.5766,29.85,30,...,0.73545,59.458598,2.537947,48.698598,0.819034,0.088841,-6039.16,3,13.183333,62.681654
3,Andhra Pradesh,2015-16,7,July,12.03,588.84,6900.0,5.8457,30.87,31,...,0.73239,78.151610,2.745967,66.121610,0.846068,0.085339,-6311.16,4,12.046667,62.047668
4,Andhra Pradesh,2015-16,8,August,12.15,588.84,6500.0,5.1350,29.22,31,...,0.73734,69.114193,2.773359,56.964193,0.824204,0.090591,-5911.16,5,11.646667,57.261467
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4345,West Bengal,2025-26,8,August,29.03,320.62,11451.0,3.8770,28.06,31,...,0.74082,28.547022,12.169801,0.000000,0.000000,0.027999,-11130.38,5,29.520000,0.241360
4346,West Bengal,2025-26,9,September,32.15,320.62,11868.0,4.4153,27.93,30,...,0.74121,31.478451,13.927009,0.000000,0.000000,0.027016,-11547.38,6,29.516667,0.241360
4347,West Bengal,2025-26,10,October,32.25,320.62,10283.0,4.3255,26.40,31,...,0.74580,32.063505,13.519673,0.000000,0.000000,0.031180,-9962.38,7,31.143333,0.000000
4348,West Bengal,2025-26,11,November,35.34,320.62,9070.0,4.1496,20.32,30,...,0.75000,29.935007,15.308881,0.000000,0.000000,0.035350,-8749.38,8,33.246667,0.000000


**Observation** : The rolling averages largely preserve the patterns observed in our original variables.

---

Saving as Flat File

In [12]:
MASTER_DF_FEATURE_ENGINEERED = '../data/processed/flat_files/MASTER_DF_FEATURE_ENGINEERED.csv'

if not os.path.isfile(MASTER_DF_FEATURE_ENGINEERED):
    df.to_csv(MASTER_DF_FEATURE_ENGINEERED, index=False)
    print(f'File saved at {MASTER_DF_FEATURE_ENGINEERED}')
else:
    print(f'File already exits at {MASTER_DF_FEATURE_ENGINEERED}')

File saved at ../data/processed/flat_files/MASTER_DF_FEATURE_ENGINEERED.csv


<br><br><br><br><br>